# MACAW-Driven Molecular Embeddings for Bioassay Modeling

This notebook demonstrates a complete workflow for drug-discovery modeling using PubChem BioAssay data and MACAW molecular embeddings. It proceeds from data ingestion to model evaluation for both classification (bioactivity outcome) and regression (potency prediction). 

## Objectives

1. Read a DataFrame (`pubchemDataDF_clean`) containing SMILES strings and bioassay targets.
2. Generate low-dimensional molecular embeddings using MACAW from SMILES strings.
3. Train and evaluate:
   - A classification model to predict `PUBCHEM_ACTIVITY_OUTCOME`.
   - A regression model to predict `LogGI50_M`.
4. Persist results and artifacts for downstream analysis using Retro-synthesis.

## Inputs and Columns of Interest

- `PUBCHEM_EXT_DATASOURCE_SMILES`: Molecular structure as a SMILES string.
- `PUBCHEM_ACTIVITY_OUTCOME`: Qualitative activity label (e.g., Active, Inactive).
- `LogGI50_M`: Mean log10 concentration for 50% growth inhibition (primary potency target).

In [2]:
import macaw
print(macaw.__version__)
from macaw import *
import tqdm as notebook_tqdm

import importlib, sys, subprocess
import pandas as pd
import numpy as np
import os

1.0.1


## Import PubChem data

In [7]:
# Data directory
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/PubchemData/bioassayDataPubchem/'

pubchemDataDF_clean = pd.read_csv(dataDir + "/pubchemData_set1.csv")
pubchemDataDF_clean

,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,LogGI50_M,LogGI50_u,LogGI50_V,IndnGI50,StddevGI50,LogTGI_M,LogTGI_u,LogTGI_V,IndnTGI,StddevTGI
0,5467463.0,B(/C=C/CCCSC1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)CO...,Inactive,0.0,-3.6144,NaN,NaN,1.0,0.0000,-3.6144,NaN,NaN,1.0,0.0000
1,5467460.0,B(/C=C/CCCSC1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)CO...,Inactive,0.0,-3.6925,NaN,NaN,1.0,0.0000,-3.6925,NaN,NaN,1.0,0.0000
2,142445632.0,B(C1=C(C(=CC=C1)C2=NC3=C(C4=C2CCCC4)C5=C(C=C3)...,Inactive,27.0,-5.5933,NaN,NaN,2.0,0.0416,-4.4568,NaN,NaN,2.0,0.6459
3,164608433.0,B(C1=C(C(=CC=C1)C2=NC3=C(C4=C2CCCC4)C5=C(NN=C5...,Inactive,32.0,-5.9467,NaN,NaN,2.0,0.0283,-4.3010,NaN,NaN,2.0,0.0000
4,167993842.0,B(C1=C(C(=CC=C1)C2=NC3=C(C=C4C(=C3C5=C2CCCC5)C...,Inactive,33.0,-5.9769,NaN,NaN,2.0,0.2475,-4.2534,NaN,NaN,2.0,0.3584
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56600,2767.0,[NH2-].[NH2-].Cl[Pt+2]Cl,Inactive,0.0,-4.0000,NaN,NaN,1.0,0.0000,-4.0000,NaN,NaN,1.0,0.0000
56601,426356.0,[NH2-].[NH2-].[NH2-].[NH2-].OS(=O)O.OS(=O)O.[Co],Inactive,0.0,-4.0000,NaN,NaN,1.0,0.0000,-4.0000,NaN,NaN,1.0,0.0000
56602,425301.0,[NH2-].[NH2-].[NH2-].[NH2-].[NH2-].Cl[Ru],Inactive,0.0,-4.0000,NaN,NaN,1.0,0.0000,-4.0000,NaN,NaN,1.0,0.0000
56603,374958.0,[NH2-].[NH2-].[NH2-].[NH2-].[NH2-].N#[O+].[O-]...,Inactive,0.0,-4.0000,NaN,NaN,1.0,0.0000,-4.0000,NaN,NaN,1.0,0.0000


In [10]:
pubchemDataForActivity = pubchemDataDF_clean.copy()

requiredCols = [
    "PUBCHEM_EXT_DATASOURCE_SMILES",
    "PUBCHEM_ACTIVITY_OUTCOME",
    "LogGI50_M",
]

missingCols = [c for c in requiredCols if c not in pubchemDataForActivity.columns]
if missingCols:
    raise ValueError(f"Missing required columns: {missingCols}")

# Clean column names and format data
pubchemDataForActivity.columns = [c.strip() for c in pubchemDataForActivity.columns]
pubchemDataForActivity["PUBCHEM_ACTIVITY_OUTCOME"] = pubchemDataForActivity["PUBCHEM_ACTIVITY_OUTCOME"].astype(str).str.strip().str.title()

# Keep only the required columns
pubchemDataForActivity = pubchemDataForActivity[requiredCols]

print(f"Dataset shape after filtering: {pubchemDataForActivity.shape}")
print(f"Columns retained: {list(pubchemDataForActivity.columns)}")
pubchemDataForActivity

Dataset shape after filtering: (56605, 3)
Columns retained: ['PUBCHEM_EXT_DATASOURCE_SMILES', 'PUBCHEM_ACTIVITY_OUTCOME', 'LogGI50_M']


,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,LogGI50_M
0,B(/C=C/CCCSC1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)CO...,Inactive,-3.6144
1,B(/C=C/CCCSC1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)CO...,Inactive,-3.6925
2,B(C1=C(C(=CC=C1)C2=NC3=C(C4=C2CCCC4)C5=C(C=C3)...,Inactive,-5.5933
3,B(C1=C(C(=CC=C1)C2=NC3=C(C4=C2CCCC4)C5=C(NN=C5...,Inactive,-5.9467
4,B(C1=C(C(=CC=C1)C2=NC3=C(C=C4C(=C3C5=C2CCCC5)C...,Inactive,-5.9769
...,...,...,...
56600,[NH2-].[NH2-].Cl[Pt+2]Cl,Inactive,-4.0000
56601,[NH2-].[NH2-].[NH2-].[NH2-].OS(=O)O.OS(=O)O.[Co],Inactive,-4.0000
56602,[NH2-].[NH2-].[NH2-].[NH2-].[NH2-].Cl[Ru],Inactive,-4.0000
56603,[NH2-].[NH2-].[NH2-].[NH2-].[NH2-].N#[O+].[O-]...,Inactive,-4.0000


## MACAW Embedding Generation

MACAW converts SMILES strings into compact, information-rich embeddings suitable for statistical learning. This section:
- Extracts valid SMILES.
- Computes MACAW embeddings via `fit_transform`.
- Aligns embeddings with downstream targets for classification and regression.

In [12]:
from macaw import *

smilesSeries = pubchemDataForActivity["PUBCHEM_EXT_DATASOURCE_SMILES"].astype(str).str.strip()
validMask = smilesSeries.notna() & smilesSeries.ne("")
dfValid = pubchemDataForActivity.loc[validMask].reset_index(drop=True)

macawModel = MACAW()
embeddings = macawModel.fit_transform(dfValid["PUBCHEM_EXT_DATASOURCE_SMILES"].tolist())
print("Embedding matrix shape:", embeddings.shape)

[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerator
[16:12:12] DEPRECATION WARNING: please use MorganGenerat

Embedding matrix shape: (56605, 15)


## Classification Task: Predicting Bioactivity Outcome

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

yCls = dfValid["PUBCHEM_ACTIVITY_OUTCOME"].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, yCls.values, test_size=0.2, random_state=42, stratify=yCls.values
)

clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42, class_weight="balanced_subsample")
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, digits=4))
print(confusion_matrix(y_test, y_pred))

## Regression Task: Predicting Potency (LogGI50_M)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

yRegFull = pd.to_numeric(dfValid["LogGI50_M"], errors="coerce")
regMask = yRegFull.notna()
XReg = embeddings[regMask]
yReg = yRegFull[regMask].values
Xr_train, Xr_test, yr_train, yr_test = train_test_split(XReg, yReg, test_size=0.2, random_state=42)

reg = RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=42)
reg.fit(Xr_train, yr_train)
yr_pred = reg.predict(Xr_test)
print("MSE:", mean_squared_error(yr_test, yr_pred))
print("MAE:", mean_absolute_error(yr_test, yr_pred))
print("R^2:", r2_score(yr_test, yr_pred))